# Automated Causal Analysis Report Generation

**Module 07 — Production Pipelines**

**Estimated time:** 15 minutes

## Learning Objectives

By the end of this notebook, you will be able to:

1. Generate a complete, standardised causal analysis report from an estimation result
2. Compute and report interpretable effect sizes alongside point estimates
3. Produce a robustness table across multiple specifications
4. Visualise posterior distributions and forest plots for technical audiences
5. Structure sensitivity analysis output for honest communication of uncertainty

## Setup

This notebook builds on the reporting standards in **Guide 02 — Reporting Causal Estimates**. We run a DiD analysis on simulated minimum wage data, then generate a full structured report including effect sizes, robustness checks, and sensitivity analysis.

In [ ]:
learning_objectives(["Generate a complete, standardised causal analysis report from an estimation result", "Compute and report interpretable effect sizes alongside point estimates", "Produce a robustness table across multiple specifications", "Visualise posterior distributions and forest plots for technical audiences", "Structure sensitivity analysis output for honest communication of uncertainty"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.formula.api as smf
import warnings
from scipy import stats
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Libraries loaded.')

## 1. Data Generation: Minimum Wage Policy

We simulate a Card & Krueger-style minimum wage study. New Jersey raises its minimum wage; Pennsylvania does not. The outcome is fast-food employment (FTE workers per store).

- **True ATT:** 2.76 FTE (Card & Krueger 1994 estimate)
- **N stores:** 400 (200 per state)
- **Periods:** Pre (t=0) and Post (t=1)

In [ ]:
# ── Simulation parameters ─────────────────────────────────────────────────────
N_STORES      = 200          # stores per state
TRUE_ATT      = 2.76         # Card & Krueger point estimate (FTE workers)
BASELINE_NJ   = 20.44        # pre-treatment NJ mean
BASELINE_PA   = 23.33        # pre-treatment PA mean
NOISE_SD      = 5.0          # within-store noise
OUTCOME_LABEL = 'FTE workers per store'


def simulate_minimum_wage(n_stores=N_STORES, true_att=TRUE_ATT, seed=42):
    """Simulate Card & Krueger minimum wage DiD dataset."""
    rng = np.random.default_rng(seed)

    store_effects_nj = rng.normal(0, 2, n_stores)   # store-level heterogeneity
    store_effects_pa = rng.normal(0, 2, n_stores)

    records = []
    for i in range(n_stores):
        for t in [0, 1]:
            # New Jersey (treated)
            y_nj = (
                BASELINE_NJ
                + store_effects_nj[i]
                + (true_att if t == 1 else 0)        # treatment effect post
                + rng.normal(0, NOISE_SD)
            )
            records.append({
                'store_id': f'NJ_{i:03d}',
                'state': 'NJ',
                'treated': 1,
                'post': t,
                'fte': y_nj,
            })
            # Pennsylvania (control)
            y_pa = (
                BASELINE_PA
                + store_effects_pa[i]
                + rng.normal(0, NOISE_SD)
            )
            records.append({
                'store_id': f'PA_{i:03d}',
                'state': 'PA',
                'treated': 0,
                'post': t,
                'fte': y_pa,
            })

    df = pd.DataFrame(records)
    df['post_treated'] = df['post'] * df['treated']   # interaction term
    return df


df = simulate_minimum_wage()

print(f'Dataset shape: {df.shape}')
print(f'States:        {df["state"].unique()}')
print(f'Periods:       {df["post"].unique()}')
print(f'Stores:        {df["store_id"].nunique()}')
print()
print('Group means:')
print(df.groupby(['state', 'post'])['fte'].mean().round(2).unstack())

## 2. Primary Estimation

We fit the standard DiD regression with store fixed effects and clustered standard errors, then extract the primary estimate along with uncertainty measures.

In [ ]:
@dataclass
class CausalEstimate:
    """Structured container for a single causal estimate."""
    label: str
    estimate: float
    se: float
    ci_lo: float
    ci_hi: float
    n_obs: int
    notes: str = ''

    @property
    def significant(self) -> bool:
        """True if 95% CI excludes zero."""
        return (self.ci_lo > 0) or (self.ci_hi < 0)

    @property
    def t_stat(self) -> float:
        return self.estimate / self.se

    @property
    def p_value(self) -> float:
        return 2 * (1 - stats.norm.cdf(abs(self.t_stat)))


def run_did(df: pd.DataFrame, label: str, formula: str) -> CausalEstimate:
    """Fit a DiD OLS model and return a structured estimate."""
    model = smf.ols(formula, data=df).fit(
        cov_type='cluster', cov_kwds={'groups': df['store_id']}
    )
    coef = model.params['post_treated']
    se   = model.bse['post_treated']
    ci   = model.conf_int().loc['post_treated'].values
    return CausalEstimate(
        label=label,
        estimate=float(coef),
        se=float(se),
        ci_lo=float(ci[0]),
        ci_hi=float(ci[1]),
        n_obs=int(model.nobs),
    )


# Primary specification: store + time fixed effects
primary = run_did(
    df,
    label='Primary (FE)',
    formula='fte ~ post_treated + C(store_id) + C(post)',
)

print('── PRIMARY ESTIMATE ─────────────────────────────────────────────────────')
print(f'  ATT estimate: {primary.estimate:+.3f} FTE')
print(f'  Standard error: {primary.se:.3f}')
print(f'  95% CI: [{primary.ci_lo:+.3f}, {primary.ci_hi:+.3f}]')
print(f'  t-statistic: {primary.t_stat:.2f}')
print(f'  p-value: {primary.p_value:.4f}')
print(f'  N observations: {primary.n_obs}')
print(f'  Significant (CI excludes 0): {primary.significant}')
print(f'  True ATT: {TRUE_ATT} FTE')

## 3. Effect Size Computation

A raw point estimate in FTE workers is hard to interpret without context. We convert it into:

- **Cohen's d** — effect in standard deviation units (cross-study comparable)
- **Percentage change** — effect relative to the pre-treatment baseline
- **Number Needed to Treat (NNT)** — for binary outcomes (omitted here, shown for reference)

These are the context-rich measures that make results communicable to policymakers.

In [ ]:
def compute_effect_sizes(
    estimate: float,
    df: pd.DataFrame,
    outcome_col: str,
    pre_mask,
) -> Dict[str, float]:
    """
    Compute a suite of interpretable effect size metrics.

    Parameters
    ----------
    estimate    : raw point estimate
    df          : full dataset
    outcome_col : name of the outcome column
    pre_mask    : boolean mask selecting pre-treatment rows
    """
    outcome_sd    = df[outcome_col].std()
    baseline_mean = df.loc[pre_mask, outcome_col].mean()

    cohens_d       = estimate / outcome_sd
    pct_change     = 100.0 * estimate / baseline_mean
    percentile_shift = (stats.norm.cdf(cohens_d) - 0.5) * 100  # percentage points

    magnitude_label = (
        'small'  if abs(cohens_d) < 0.5 else
        'medium' if abs(cohens_d) < 0.8 else
        'large'
    )

    return {
        'raw_estimate':      estimate,
        'outcome_sd':        outcome_sd,
        'baseline_mean':     baseline_mean,
        'cohens_d':          cohens_d,
        'magnitude_label':   magnitude_label,
        'pct_change':        pct_change,
        'percentile_shift':  percentile_shift,
    }


pre_mask = df['post'] == 0
effect_sizes = compute_effect_sizes(primary.estimate, df, 'fte', pre_mask)

print('── EFFECT SIZES ─────────────────────────────────────────────────────────')
print(f'  Raw estimate:        {effect_sizes["raw_estimate"]:+.3f} FTE workers per store')
print(f'  Baseline mean (pre): {effect_sizes["baseline_mean"]:.2f} FTE')
print(f'  Outcome SD:          {effect_sizes["outcome_sd"]:.2f} FTE')
print()
print(f'  Cohen\'s d:           {effect_sizes["cohens_d"]:.3f}  ({effect_sizes["magnitude_label"]})')
print(f'  % of baseline:       {effect_sizes["pct_change"]:+.1f}%')
print(f'  Percentile shift:    +{effect_sizes["percentile_shift"]:.1f} percentile points')
print()
print(
    f'  Interpretation: The minimum wage increase raised employment by '
    f'{effect_sizes["raw_estimate"]:.2f} FTE workers per store '
    f'({effect_sizes["pct_change"]:+.1f}% of the pre-period baseline), '
    f'a {effect_sizes["magnitude_label"]} effect (d = {effect_sizes["cohens_d"]:.2f}).'
)

## 4. Robustness Table

A single specification is never sufficient. We report the primary result alongside five alternative specifications:

1. **No store FE** — pooled DiD without unit fixed effects
2. **No time FE** — drops period fixed effects
3. **State-level clustering** — aggregates to state level
4. **HC1 robust SE** — heteroskedasticity-robust standard errors
5. **Treated units only** — restricts to New Jersey stores (placebo sanity check)

If the result is robust, all specifications should show similar point estimates.

In [ ]:
def run_did_hc1(df, label, formula):
    """DiD with HC1 robust SEs (no clustering)."""
    model = smf.ols(formula, data=df).fit(cov_type='HC1')
    coef = model.params['post_treated']
    se   = model.bse['post_treated']
    ci   = model.conf_int().loc['post_treated'].values
    return CausalEstimate(
        label=label, estimate=float(coef), se=float(se),
        ci_lo=float(ci[0]), ci_hi=float(ci[1]), n_obs=int(model.nobs),
    )


# --- Specification 1: Primary -----------------------------------------------
spec_primary = primary  # already computed above

# --- Specification 2: No store FE -------------------------------------------
spec_no_store_fe = run_did(
    df, label='No store FE',
    formula='fte ~ post_treated + treated + C(post)',
)

# --- Specification 3: No time FE --------------------------------------------
spec_no_time_fe = run_did(
    df, label='No time FE',
    formula='fte ~ post_treated + C(store_id)',
)

# --- Specification 4: HC1 robust SE -----------------------------------------
spec_hc1 = run_did_hc1(
    df, label='HC1 robust SE',
    formula='fte ~ post_treated + C(store_id) + C(post)',
)

# --- Specification 5: Treated units only (within-group trend) ---------------
df_nj = df[df['treated'] == 1].copy()
spec_nj_only = run_did_hc1(
    df_nj, label='Treated stores only',
    formula='fte ~ post_treated + C(store_id)',
)


all_specs = [
    spec_primary,
    spec_no_store_fe,
    spec_no_time_fe,
    spec_hc1,
    spec_nj_only,
]

# --- Print robustness table -------------------------------------------------
print('Table 1: Robustness Checks — Effect of Minimum Wage on FTE Employment')
print('=' * 72)
header = f'{"Specification":<28} {"Estimate":>8} {"95% CI":>20} {"N":>6} {"Sig":>5}'
print(header)
print('-' * 72)
for s in all_specs:
    ci_str  = f'[{s.ci_lo:+.3f}, {s.ci_hi:+.3f}]'
    sig_str = 'Yes' if s.significant else 'No'
    bold    = '*** ' if s.label == 'Primary (FE)' else '    '
    print(f'{bold}{s.label:<24} {s.estimate:>+8.3f} {ci_str:>20} {s.n_obs:>6} {sig_str:>5}')
print('-' * 72)
n_sig = sum(s.significant for s in all_specs)
print(f'Significant in {n_sig}/{len(all_specs)} specifications.')
print('*** = primary specification')

## 5. Pre-Trend Test

Before acting on the DiD estimate, we need to verify that the parallel trends assumption is plausible. We test whether treated and control stores were on diverging trends in the pre-treatment period.

In our two-period setting there is only one pre-period, so we test whether the baseline difference between groups is within an acceptable range (a balance check rather than a trend test).

In [ ]:
def check_balance(df: pd.DataFrame, outcome_col: str, group_col: str, pre_mask) -> Dict:
    """
    Check whether treated and control groups have similar pre-treatment outcomes.

    Returns a dict with means, standardised difference, and a pass/fail flag.
    """
    pre_df = df[pre_mask].copy()

    treated_mean = pre_df.loc[pre_df[group_col] == 1, outcome_col].mean()
    control_mean = pre_df.loc[pre_df[group_col] == 0, outcome_col].mean()
    pooled_sd    = pre_df[outcome_col].std()

    std_diff = (treated_mean - control_mean) / pooled_sd

    # Two-sample t-test for baseline difference
    treated_vals = pre_df.loc[pre_df[group_col] == 1, outcome_col].values
    control_vals = pre_df.loc[pre_df[group_col] == 0, outcome_col].values
    t_stat, p_val = stats.ttest_ind(treated_vals, control_vals)

    # Parallel trends: standardised difference < 0.1 is convention for balance
    balanced = abs(std_diff) < 0.5  # lenient threshold for a 2-period DiD

    return {
        'treated_mean': treated_mean,
        'control_mean': control_mean,
        'raw_diff':     treated_mean - control_mean,
        'std_diff':     std_diff,
        'baseline_t':   t_stat,
        'baseline_p':   p_val,
        'balanced':     balanced,
    }


balance = check_balance(df, 'fte', 'treated', pre_mask)

print('── PRE-TREATMENT BALANCE CHECK ──────────────────────────────────────────')
print(f'  NJ (treated) pre-mean:  {balance["treated_mean"]:.2f} FTE')
print(f'  PA (control) pre-mean:  {balance["control_mean"]:.2f} FTE')
print(f'  Raw pre-period gap:     {balance["raw_diff"]:+.2f} FTE')
print(f'  Standardised diff:      {balance["std_diff"]:+.3f}')
print(f'  Baseline t-test:        t = {balance["baseline_t"]:.2f}, p = {balance["baseline_p"]:.4f}')
print()
status = 'PASS' if balance['balanced'] else 'WARN'
print(f'  Balance check: {status}')
print()
print('  Note: A significant pre-period difference does not directly violate parallel')
print('  trends — it only means groups start at different levels. Parallel trends')
print('  requires that trajectories would have been parallel absent treatment.')

## 6. Sensitivity Analysis

Sensitivity analysis answers: "What would have to be true for my conclusion to be wrong?"

For DiD, we examine how the conclusion changes under hypothetical parallel trends violations of increasing size. This follows the Roth (2022) approach: if there were a pre-trend of size `δ`, the treatment effect would be biased by approximately `δ`.

In [ ]:
def sensitivity_to_pretrend(
    estimate: float,
    se: float,
    violations: List[float],
    alpha: float = 0.05,
) -> pd.DataFrame:
    """
    Show how the treatment effect conclusion changes under pre-trend violations.

    For each hypothetical violation size δ:
      - adjusted_estimate = estimate - δ  (worst-case direction)
      - adjusted CI = [estimate - δ ± z * se]

    Parameters
    ----------
    estimate   : primary point estimate
    se         : clustered SE of primary estimate
    violations : list of violation sizes to consider
    alpha      : significance level
    """
    z_crit = stats.norm.ppf(1 - alpha / 2)
    rows = []
    for delta in violations:
        adj_est = estimate - delta
        adj_lo  = adj_est - z_crit * se
        adj_hi  = adj_est + z_crit * se
        sig     = (adj_lo > 0) or (adj_hi < 0)
        rows.append({
            'violation_δ':   delta,
            'adj_estimate':  adj_est,
            'adj_ci_lo':     adj_lo,
            'adj_ci_hi':     adj_hi,
            'still_sig':     sig,
        })
    return pd.DataFrame(rows)


violations = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
sensitivity_df = sensitivity_to_pretrend(primary.estimate, primary.se, violations)

print('── SENSITIVITY ANALYSIS: Pre-Trend Violations ───────────────────────────')
print(f'  Primary estimate: {primary.estimate:+.3f} FTE (SE = {primary.se:.3f})')
print()
print(f'  {"Violation δ":>12} {"Adj. Estimate":>14} {"95% CI":>22} {"Still Sig?":>12}')
print('  ' + '-' * 62)
for _, row in sensitivity_df.iterrows():
    ci_str  = f'[{row["adj_ci_lo"]:+.3f}, {row["adj_ci_hi"]:+.3f}]'
    sig_str = 'Yes' if row['still_sig'] else 'NO — flips/loses sig'
    marker  = ' ← primary' if row['violation_δ'] == 0.0 else ''
    print(f'  {row["violation_δ"]:>12.1f} {row["adj_estimate"]:>+14.3f} {ci_str:>22} {sig_str:>12}{marker}')

# Find the violation size that would make the result insignificant
max_violation = sensitivity_df.loc[sensitivity_df['still_sig'], 'violation_δ'].max()
print()
print(f'  Conclusion: The result remains significant for violations up to δ ≈ {max_violation:.1f} FTE.')
print(f'  A violation greater than {max_violation:.1f} FTE would be needed to overturn the conclusion.')

## 7. Visualisations

### 7a. Forest Plot (Robustness Visualisation)

The forest plot is the visual equivalent of the robustness table. All specifications should cluster on the same side of zero.

In [ ]:
def forest_plot(
    estimates: List[CausalEstimate],
    title: str = 'Treatment Effect Estimates',
    xlabel: str = 'Treatment Effect',
    ax=None,
):
    """
    Forest plot for a list of CausalEstimate objects.

    The first estimate is treated as the primary specification (steelblue, larger dot).
    All others are secondary specifications (gray).
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, len(estimates) * 0.7 + 1.5))

    for i, est in enumerate(estimates):
        is_primary = i == 0
        color = 'steelblue' if is_primary else '#888'
        size  = 100 if is_primary else 60
        lw    = 2.5 if is_primary else 1.5

        ax.plot([est.ci_lo, est.ci_hi], [i, i], '-', color=color, lw=lw, alpha=0.85)
        ax.scatter(est.estimate, i, color=color, s=size, zorder=5)

        # Annotate with estimate value
        ax.text(
            est.ci_hi + 0.05, i,
            f'{est.estimate:+.2f}',
            va='center', fontsize=9,
            color=color, fontweight='bold' if is_primary else 'normal',
        )

    # Zero line
    ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5, label='No effect')

    ax.set_yticks(range(len(estimates)))
    ax.set_yticklabels([e.label for e in estimates])
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

    return ax


fig, ax = plt.subplots(figsize=(11, 5))
forest_plot(
    all_specs,
    title='Fig 1: Treatment Effect of Minimum Wage on FTE Employment — Robustness',
    xlabel='Effect on FTE Workers per Store',
    ax=ax,
)
ax.axvline(TRUE_ATT, color='green', lw=1.5, ls=':', alpha=0.7, label=f'True ATT = {TRUE_ATT}')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()

### 7b. Sensitivity Analysis Plot

This plot shows how the adjusted estimate and its confidence interval change as the hypothetical pre-trend violation grows. The shaded region is the 95% CI. Once the CI crosses zero, the conclusion is overturned.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

violations_fine = np.linspace(0, 3.5, 100)
sens_fine = sensitivity_to_pretrend(primary.estimate, primary.se, violations_fine.tolist())

ax.fill_between(
    sens_fine['violation_δ'],
    sens_fine['adj_ci_lo'],
    sens_fine['adj_ci_hi'],
    alpha=0.20, color='steelblue', label='95% CI (adjusted)',
)
ax.plot(
    sens_fine['violation_δ'], sens_fine['adj_estimate'],
    '-', color='steelblue', lw=2, label='Adjusted estimate',
)
ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5, label='No effect')

# Mark where CI first crosses zero
flip_idx = (sens_fine['adj_ci_lo'] <= 0).idxmax()
flip_x   = sens_fine.loc[flip_idx, 'violation_δ']
ax.axvline(flip_x, color='red', lw=1.2, ls='--', alpha=0.6,
           label=f'CI crosses 0 at δ = {flip_x:.2f}')

ax.set_xlabel('Pre-trend violation size δ (FTE units)')
ax.set_ylabel('Adjusted treatment effect estimate')
ax.set_title(
    'Fig 2: Sensitivity to Pre-Trend Violations\n'
    '(Roth-style: conclusion overturned when CI crosses zero)',
    fontweight='bold',
)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 7c. Means-Over-Time Plot (DiD Visualisation)

The canonical DiD visualisation shows group means before and after treatment, with the counterfactual trajectory for the treated group superimposed.

In [ ]:
def did_means_plot(df, outcome, treated_col, post_col, ax=None):
    """Plot group means over time with counterfactual for treated group."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    means = df.groupby([post_col, treated_col])[outcome].mean().unstack()
    # means.columns: [0 = control, 1 = treated]

    times = [0, 1]

    # Observed trajectories
    ax.plot(times, means[1].values, 'o-', color='steelblue',
            lw=2.5, ms=10, label='New Jersey (treated)')
    ax.plot(times, means[0].values, 's--', color='gray',
            lw=2.5, ms=10, label='Pennsylvania (control)')

    # Counterfactual: NJ pre-level + PA change
    nj_pre = means.loc[0, 1]
    pa_change = means.loc[1, 0] - means.loc[0, 0]
    nj_counterfactual = nj_pre + pa_change

    ax.plot([0, 1], [nj_pre, nj_counterfactual], 'o:',
            color='steelblue', lw=1.5, ms=10, alpha=0.5,
            label='NJ counterfactual (no wage increase)')

    # ATT arrow
    nj_post  = means.loc[1, 1]
    ax.annotate(
        '', xy=(1, nj_post), xytext=(1, nj_counterfactual),
        arrowprops=dict(arrowstyle='<->', color='darkgreen', lw=2),
    )
    ax.text(
        1.03, (nj_post + nj_counterfactual) / 2,
        f'ATT = {nj_post - nj_counterfactual:+.2f}',
        va='center', color='darkgreen', fontsize=10, fontweight='bold',
    )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Pre-treatment\n(Feb 1992)', 'Post-treatment\n(Nov 1992)'])
    ax.set_ylabel('Mean FTE Workers per Store')
    ax.set_title('Fig 3: Difference-in-Differences — Means Over Time', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    return ax


fig, ax = plt.subplots(figsize=(9, 5))
did_means_plot(df, 'fte', 'treated', 'post', ax=ax)
plt.tight_layout()
plt.show()

## 8. Automated Report Generation

We combine all the outputs above into a structured report function that produces a single printable summary. In production, this function would write to a JSON manifest, a PDF, or a HTML report file.

In [ ]:
def generate_causal_report(
    primary:        CausalEstimate,
    all_specs:      List[CausalEstimate],
    effect_sizes:   Dict,
    balance:        Dict,
    sensitivity_df: pd.DataFrame,
    design:         str,
    outcome_label:  str,
    treatment_desc: str,
    true_att:       Optional[float] = None,
) -> str:
    """
    Generate a complete causal analysis report as a formatted string.

    All inputs come from the estimation, effect size, and diagnostic functions
    defined earlier in this notebook.
    """
    z_crit  = 1.96
    run_id  = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
    n_sig   = sum(s.significant for s in all_specs)

    # Find the maximum violation that keeps the result significant
    sig_rows = sensitivity_df[sensitivity_df['still_sig']]
    max_robustness = sig_rows['violation_δ'].max() if not sig_rows.empty else 0.0

    sep = '=' * 72
    thin = '-' * 72

    lines = [
        sep,
        'CAUSAL ANALYSIS REPORT',
        sep,
        f'Run ID:       {run_id}',
        f'Design:       {design}',
        f'Treatment:    {treatment_desc}',
        f'Outcome:      {outcome_label}',
        '',
        thin,
        '1. RESULTS SUMMARY',
        thin,
        '',
        f'  Point estimate:  {primary.estimate:+.3f} {outcome_label}',
        f'  Standard error:  {primary.se:.3f}',
        f'  95% CI:          [{primary.ci_lo:+.3f}, {primary.ci_hi:+.3f}]',
        f'  t-statistic:     {primary.t_stat:.2f}',
        f'  p-value:         {primary.p_value:.4f}',
        f'  N observations:  {primary.n_obs}',
        f'  Significant:     {"Yes" if primary.significant else "No"}',
    ]

    if true_att is not None:
        lines.append(f'  True ATT:        {true_att:+.3f} (simulation ground truth)')

    lines += [
        '',
        '  Complete results sentence:',
        f'  The {design} estimate suggests that {treatment_desc} changed',
        f'  {outcome_label} by {primary.estimate:+.3f} units',
        f'  (95% CI: {primary.ci_lo:+.3f}, {primary.ci_hi:+.3f}; N = {primary.n_obs}).',
        f'  This is a {effect_sizes["magnitude_label"]} effect',
        f'  (d = {effect_sizes["cohens_d"]:+.3f}),',
        f'  equivalent to {effect_sizes["pct_change"]:+.1f}% of the pre-treatment baseline.',
        '',
        thin,
        '2. EFFECT SIZES',
        thin,
        '',
        f'  Cohen\'s d:          {effect_sizes["cohens_d"]:+.3f} ({effect_sizes["magnitude_label"]})',
        f'  % of baseline:      {effect_sizes["pct_change"]:+.1f}%',
        f'  Baseline mean:      {effect_sizes["baseline_mean"]:.2f}',
        f'  Outcome SD:         {effect_sizes["outcome_sd"]:.2f}',
        '',
        thin,
        '3. ASSUMPTION DIAGNOSTICS',
        thin,
        '',
        f'  Pre-period balance:',
        f'    Treated mean:      {balance["treated_mean"]:.2f}',
        f'    Control mean:      {balance["control_mean"]:.2f}',
        f'    Std. difference:   {balance["std_diff"]:+.3f}',
        f'    Balance status:    {"PASS" if balance["balanced"] else "WARN"}',
        '',
        thin,
        '4. ROBUSTNESS',
        thin,
        '',
        f'  Primary specification: {primary.estimate:+.3f} (CI: {primary.ci_lo:+.3f}, {primary.ci_hi:+.3f})',
        f'  Significant in {n_sig}/{len(all_specs)} alternative specifications.',
        '',
    ]

    for s in all_specs[1:]:
        sig_str = 'sig' if s.significant else 'not sig'
        lines.append(
            f'    {s.label:<28}: {s.estimate:+.3f} [{s.ci_lo:+.3f}, {s.ci_hi:+.3f}] ({sig_str})'
        )

    lines += [
        '',
        thin,
        '5. SENSITIVITY ANALYSIS',
        thin,
        '',
        f'  Conclusion stable for parallel-trends violations up to δ = {max_robustness:.1f}.',
        f'  A violation larger than {max_robustness:.1f} {outcome_label} would overturn',
        f'  the conclusion (CI would include zero).',
        '',
        thin,
        '6. LIMITATIONS',
        thin,
        '',
        '  - Parallel trends assumption is untestable from two periods.',
        '  - ATT is local to New Jersey fast-food stores (external validity limited).',
        '  - Estimation uses OLS — Bayesian estimation would provide posterior',
        '    probability P(τ > 0) and a full uncertainty distribution.',
        '  - Simulation ground truth is known; real-world applications cannot verify.',
        '',
        sep,
        'END OF REPORT',
        sep,
    ]

    return '\n'.join(lines)


report = generate_causal_report(
    primary        = primary,
    all_specs      = all_specs,
    effect_sizes   = effect_sizes,
    balance        = balance,
    sensitivity_df = sensitivity_df,
    design         = 'Difference-in-Differences',
    outcome_label  = OUTCOME_LABEL,
    treatment_desc = 'New Jersey minimum wage increase (April 1992)',
    true_att       = TRUE_ATT,
)

print(report)

## 9. Comprehensive Summary Dashboard

Finally, we combine the three key figures into a single dashboard suitable for a technical presentation.

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

ax_did    = fig.add_subplot(gs[0, 0])
ax_forest = fig.add_subplot(gs[0, 1])
ax_sens   = fig.add_subplot(gs[1, 0])
ax_text   = fig.add_subplot(gs[1, 1])

# ── Panel A: DiD means over time ────────────────────────────────────────────
did_means_plot(df, 'fte', 'treated', 'post', ax=ax_did)
ax_did.set_title('A. DiD Means Over Time', fontweight='bold')

# ── Panel B: Forest plot ─────────────────────────────────────────────────────
forest_plot(
    all_specs,
    title='B. Robustness: Multiple Specifications',
    xlabel='Effect on FTE per Store',
    ax=ax_forest,
)
ax_forest.axvline(TRUE_ATT, color='green', lw=1.2, ls=':', alpha=0.6)

# ── Panel C: Sensitivity analysis ───────────────────────────────────────────
ax_sens.fill_between(
    sens_fine['violation_δ'],
    sens_fine['adj_ci_lo'],
    sens_fine['adj_ci_hi'],
    alpha=0.20, color='steelblue',
)
ax_sens.plot(
    sens_fine['violation_δ'], sens_fine['adj_estimate'],
    '-', color='steelblue', lw=2,
)
ax_sens.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
ax_sens.axvline(flip_x, color='red', lw=1.2, ls='--', alpha=0.6)
ax_sens.set_xlabel('Pre-trend violation δ')
ax_sens.set_ylabel('Adjusted estimate')
ax_sens.set_title('C. Sensitivity to Pre-Trend Violations', fontweight='bold')
ax_sens.grid(alpha=0.3)

# ── Panel D: Key numbers table ───────────────────────────────────────────────
ax_text.axis('off')
summary_text = (
    f"D. Key Results Summary\n"
    f"{'─' * 36}\n"
    f"Design:         DiD (NJ vs PA)\n"
    f"Treatment:      NJ min wage hike\n"
    f"Outcome:        FTE per store\n"
    f"\n"
    f"ATT estimate:   {primary.estimate:+.3f} FTE\n"
    f"SE:             {primary.se:.3f}\n"
    f"95% CI:         [{primary.ci_lo:+.2f}, {primary.ci_hi:+.2f}]\n"
    f"True ATT:       {TRUE_ATT:+.3f} (sim truth)\n"
    f"\n"
    f"Cohen's d:      {effect_sizes['cohens_d']:+.3f} ({effect_sizes['magnitude_label']})\n"
    f"% of baseline:  {effect_sizes['pct_change']:+.1f}%\n"
    f"\n"
    f"Robust in:      {sum(s.significant for s in all_specs)}/{len(all_specs)} specs\n"
    f"Sens. limit:    δ = {max_violation:.1f} FTE\n"
    f"\n"
    f"Balance:        {'PASS' if balance['balanced'] else 'WARN'}\n"
)
ax_text.text(
    0.05, 0.95, summary_text,
    transform=ax_text.transAxes,
    fontsize=10, verticalalignment='top',
    fontfamily='monospace',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.6),
)

fig.suptitle(
    'Causal Analysis Report Dashboard\n'
    'Minimum Wage Effect on Fast-Food Employment (Card & Krueger 1994 Simulation)',
    fontsize=13, fontweight='bold', y=1.01,
)
plt.show()

## Exercise

The `generate_causal_report()` function currently outputs plain text. Extend it for an RDD analysis:

1. Replace the DiD simulation with the scholarship RDD from Module 05 (cutoff = 700, outcome = GPA). You'll need to adapt `run_did()` to a local linear estimator.
2. Add a "bandwidth sensitivity" section to the report text, using the `rdd_estimate()` function from Module 05 across three bandwidths: 50, 100, and 150.
3. Replace the sensitivity analysis section with an RDD-specific sensitivity table showing results under the donut RDD (excluding observations within ±5 of the cutoff) and under placebo cutoffs at 690 and 710.

Verify that the report's "Conclusion" section accurately reflects whether the result is significant in all three bandwidth specifications.

## Summary

| Step | Function | What It Produces |
|------|----------|------------------|
| Estimate | `run_did()` | `CausalEstimate` with SE and CI |
| Effect sizes | `compute_effect_sizes()` | Cohen's d, % change, percentile shift |
| Robustness | Multiple `run_did()` calls | Table of estimates across specifications |
| Diagnostics | `check_balance()` | Pre-period balance assessment |
| Sensitivity | `sensitivity_to_pretrend()` | Conclusion stability under violations |
| Report | `generate_causal_report()` | Full structured report |
| Figures | `forest_plot()`, `did_means_plot()` | Publication-ready visuals |

---

**Previous:** [01 — Model Selection Workflow](01_model_selection_workflow.ipynb)

**Next:** [Module 07 Exercises](../exercises/01_pipeline_self_check.py)

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])